In [1]:
"""
L1-regularized decoder manifold diagnostics (TRUE vs PERM)

- Fits logistic regression with L1 penalty across a sweep of C values
- For each fit:
    * training accuracy
    * number of nonzero coefficients (sparsity)
    * Fisher sharpness ratio: (axis Fisher) / (mean random-direction Fisher)
      computed at the fitted beta (fixed), using the empirical Fisher metric

Safe for ~40k neurons, ~100–300 fits, modest C grid.
"""

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ============================================================
# PATHS
# ============================================================
VIT_PATH    = '/home/maria/ProjectionSort/data/google_vit-base-patch16-224_embeddings_logits.pkl'
NEURAL_PATH = '/home/maria/ProjectionSort/data/hybrid_neural_responses_reduced.npy'
AREAS_PATH  = '/home/maria/ProjectionSort/data/brain_area.npy'

# ============================================================
# LOAD DATA
# ============================================================
vit   = np.load(VIT_PATH, allow_pickle=True)['natural_scenes']   # (images, 1000)
R     = np.load(NEURAL_PATH).T                                   # (images, neurons)
areas = np.load(AREAS_PATH, allow_pickle=True)

print("Images:", vit.shape[0])
print("Neurons:", R.shape[1])

# ============================================================
# ANIMATE / INANIMATE LABEL
# ============================================================
top1 = np.argmax(vit, axis=1)
y_true = (top1 <= 397).astype(int)
print("Animate fraction:", y_true.mean())

# ============================================================
# CONFIG
# ============================================================
N_FITS = 200          # 1 true + (N_FITS-1) perms
N_RAND_DIRS = 100     # random probe directions for Fisher baseline
SEED = 0

# L1 strength sweep: smaller C => stronger regularization => sparser
C_GRID = np.array([0.02, 0.05, 0.1, 0.2, 0.5, 1.0])

rng = np.random.default_rng(SEED)

# ============================================================
# STANDARDIZE ONCE (IMPORTANT!)
# ============================================================
scaler = StandardScaler(with_mean=True, with_std=True)
X = scaler.fit_transform(R)      # (images, neurons)

# ============================================================
# HELPERS
# ============================================================
def fit_l1_logreg(X, y, C):
    clf = LogisticRegression(
        penalty="l1",
        C=C,
        solver="saga",
        max_iter=5000,
        tol=1e-4,
        n_jobs=-1
    )
    clf.fit(X, y)
    return clf

def fisher_quadratic_form_empirical(X, beta, intercept):
    """
    Returns a function q(v) = v^T F v for empirical Fisher F at fitted (beta, intercept),
    where F = (1/n) X^T diag(p(1-p)) X  for logistic regression.
    """
    z = X @ beta + intercept
    p = 1.0 / (1.0 + np.exp(-z))
    w = p * (1 - p)              # (n_samples,)
    # We'll compute q(v) efficiently without building F:
    # q(v) = mean( w * (Xv)^2 )
    def q(v):
        Xv = X @ v
        return float(np.mean(w * (Xv ** 2)))
    return q, p

def random_unit_vectors(d, m, rng):
    V = rng.normal(size=(m, d)).astype(np.float32)
    V /= (np.linalg.norm(V, axis=1, keepdims=True) + 1e-12)
    return V

# ============================================================
# MAIN LOOP
# ============================================================
results = {
    "C_grid": C_GRID,
    "acc_true": np.zeros(len(C_GRID)),
    "nnz_true": np.zeros(len(C_GRID), dtype=int),
    "sharp_true": np.zeros(len(C_GRID)),
    "acc_perm_mean": np.zeros(len(C_GRID)),
    "acc_perm_std": np.zeros(len(C_GRID)),
    "nnz_perm_mean": np.zeros(len(C_GRID)),
    "nnz_perm_std": np.zeros(len(C_GRID)),
    "sharp_perm_mean": np.zeros(len(C_GRID)),
    "sharp_perm_std": np.zeros(len(C_GRID)),
}

print("\n=== Running L1 sweep ===")
for ci, C in enumerate(C_GRID):
    print(f"\n--- C = {C} ---")

    accs, nnzs, sharps = [], [], []

    for k in range(N_FITS):
        yk = y_true.copy() if k == 0 else rng.permutation(y_true)

        clf = fit_l1_logreg(X, yk, C=C)
        beta = clf.coef_.ravel()
        b0 = float(clf.intercept_.ravel()[0])

        # training accuracy
        acc = float(clf.score(X, yk))

        # sparsity
        nnz = int(np.sum(np.abs(beta) > 1e-12))

        # Fisher sharpness ratio
        q, p = fisher_quadratic_form_empirical(X, beta, b0)

        # axis direction = fitted beta normalized (if all zeros, sharpness undefined)
        if nnz == 0 or np.linalg.norm(beta) < 1e-12:
            sharp = np.nan
        else:
            v_axis = beta / (np.linalg.norm(beta) + 1e-12)
            axis_q = q(v_axis)

            Vrand = random_unit_vectors(X.shape[1], N_RAND_DIRS, rng)
            rand_qs = np.array([q(v) for v in Vrand], dtype=np.float64)
            sharp = axis_q / (rand_qs.mean() + 1e-12)

        accs.append(acc); nnzs.append(nnz); sharps.append(sharp)

        if k in (0, 1, 2):
            tag = "TRUE" if k == 0 else "PERM"
            print(f"  {tag:4s} acc={acc:.3f} nnz={nnz:5d} sharp={sharp:.2f}")

    accs = np.array(accs)
    nnzs = np.array(nnzs)
    sharps = np.array(sharps, dtype=np.float64)

    # TRUE row (k=0)
    results["acc_true"][ci] = accs[0]
    results["nnz_true"][ci] = nnzs[0]
    results["sharp_true"][ci] = sharps[0]

    # PERM rows (k>=1)
    results["acc_perm_mean"][ci] = accs[1:].mean()
    results["acc_perm_std"][ci]  = accs[1:].std()
    results["nnz_perm_mean"][ci] = nnzs[1:].mean()
    results["nnz_perm_std"][ci]  = nnzs[1:].std()

    # sharpness: ignore NaNs (can happen when beta all zeros)
    perm_sharp = sharps[1:]
    perm_sharp = perm_sharp[np.isfinite(perm_sharp)]
    results["sharp_perm_mean"][ci] = perm_sharp.mean() if len(perm_sharp) else np.nan
    results["sharp_perm_std"][ci]  = perm_sharp.std()  if len(perm_sharp) else np.nan

# ============================================================
# SAVE
# ============================================================
np.savez("l1_fisher_sweep_results.npz", **results)
print("\nSaved: l1_fisher_sweep_results.npz")
print("Done.")


Images: 118
Neurons: 39209
Animate fraction: 0.5338983050847458

=== Running L1 sweep ===

--- C = 0.02 ---
  TRUE acc=0.534 nnz=    0 sharp=nan
  PERM acc=0.534 nnz=    0 sharp=nan
  PERM acc=0.534 nnz=    0 sharp=nan

--- C = 0.05 ---
  TRUE acc=0.814 nnz=    9 sharp=2.60
  PERM acc=0.568 nnz=    4 sharp=1.20
  PERM acc=0.551 nnz=    5 sharp=1.36


KeyboardInterrupt: 

In [2]:
"""
FAST L1 Fisher sweep (support-restricted)

Key optimizations:
- Fisher computed only on active support
- Fewer permutations
- Fewer random probe directions
- Early stopping when beta == 0
"""
#https://chatgpt.com/g/g-p-676c80353b988191819d6d02aca806d6/c/69555d06-1764-832d-9297-0e976ccd892a

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ============================================================
# CONFIG
# ============================================================
N_PERM = 30          # was ~200
N_RAND_DIRS = 20     # was 100
C_GRID = [0.02, 0.05, 0.1, 0.2, 0.5, 1.0]
SEED = 0

rng = np.random.default_rng(SEED)

# ============================================================
# LOAD
# ============================================================
vit = np.load(VIT_PATH, allow_pickle=True)['natural_scenes']
R   = np.load(NEURAL_PATH).T

top1 = np.argmax(vit, axis=1)
y_true = (top1 <= 397).astype(int)

scaler = StandardScaler()
X = scaler.fit_transform(R)

# ============================================================
# HELPERS
# ============================================================
def fit_l1(X, y, C):
    clf = LogisticRegression(
        penalty="l1",
        solver="saga",
        C=C,
        max_iter=3000,
        tol=1e-3,
        n_jobs=-1
    )
    clf.fit(X, y)
    return clf

def fisher_sharpness_on_support(Xs, beta_s, intercept):
    """
    Fisher sharpness restricted to active support
    """
    z = Xs @ beta_s + intercept
    p = 1 / (1 + np.exp(-z))
    w = p * (1 - p)

    # axis direction
    v_axis = beta_s / (np.linalg.norm(beta_s) + 1e-12)
    axis_q = np.mean(w * (Xs @ v_axis) ** 2)

    # random directions in support
    V = rng.normal(size=(N_RAND_DIRS, len(beta_s)))
    V /= np.linalg.norm(V, axis=1, keepdims=True)

    rand_q = np.mean(
        [np.mean(w * (Xs @ v) ** 2) for v in V]
    )

    return axis_q / (rand_q + 1e-12)

# ============================================================
# MAIN
# ============================================================
results = []

for C in C_GRID:
    print(f"\n=== C = {C} ===")

    sharps_true = []
    sharps_perm = []

    for k in range(N_PERM + 1):
        y = y_true if k == 0 else rng.permutation(y_true)

        clf = fit_l1(X, y, C)
        beta = clf.coef_.ravel()
        b0 = clf.intercept_[0]

        support = np.abs(beta) > 1e-8
        nnz = support.sum()

        if nnz == 0:
            sharp = np.nan
        else:
            Xs = X[:, support]
            beta_s = beta[support]
            sharp = fisher_sharpness_on_support(Xs, beta_s, b0)

        if k == 0:
            sharps_true.append(sharp)
            print(f" TRUE nnz={nnz:4d} sharp={sharp:.2f}")
        else:
            sharps_perm.append(sharp)

    results.append({
        "C": C,
        "sharp_true": np.nanmean(sharps_true),
        "sharp_perm_mean": np.nanmean(sharps_perm),
        "sharp_perm_std": np.nanstd(sharps_perm),
    })

np.save("l1_fisher_fast_results.npy", results)
print("\nSaved l1_fisher_fast_results.npy")



=== C = 0.02 ===
 TRUE nnz=   0 sharp=nan


/tmp/ipykernel_2821792/724147830.py:110: RuntimeWarning: Mean of empty slice
  "sharp_true": np.nanmean(sharps_true),
/tmp/ipykernel_2821792/724147830.py:111: RuntimeWarning: Mean of empty slice
  "sharp_perm_mean": np.nanmean(sharps_perm),
/home/maria/global_venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,



=== C = 0.05 ===
 TRUE nnz=  17 sharp=3.55

=== C = 0.1 ===
 TRUE nnz= 199 sharp=12.07

=== C = 0.2 ===
 TRUE nnz= 616 sharp=26.26

=== C = 0.5 ===
 TRUE nnz=1827 sharp=58.10

=== C = 1.0 ===
 TRUE nnz=3782 sharp=106.92

Saved l1_fisher_fast_results.npy
